<a href="https://colab.research.google.com/github/siregarmr/rupiah-reserve-exhaustion/blob/main/Rupiah_Reserve_Exhaustion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# RUPIAH RESERVE EXHAUSTION SIMULATOR (RRES)
#
# Notes: This is a Monte Carlo simulation of Indonesia's FX reserve adequacy.
# Data: BPS API (trade), World Bank API (inflation, debt, GDP).
# ==============================================================================

# ------------------------------------------------------------------------------
# Preamble
# ------------------------------------------------------------------------------
!pip install wbgapi --quiet

import pandas as pd
import numpy as np
import datetime
import warnings
import matplotlib.pyplot as plt
import os
import csv
import requests
import time

import wbgapi as wb

# Suppress pip dependency resolver warnings (harmless for our usage)
warnings.filterwarnings('ignore', message='.*dependency resolver.*')

# ------------------------------------------------------------------------------
# BPS API Endpoints and API Key Input
# ------------------------------------------------------------------------------
BPS_BASE_URL = "https://webapi.bps.go.id/v1/api"

try:
    from google.colab import userdata
    BPS_API_KEY = userdata.get('BPS_API_KEY')
    print("BPS API key loaded from Colab secrets.")
except Exception:
    from getpass import getpass
    BPS_API_KEY = getpass('Enter your BPS API key (press Enter to skip): ')
    if BPS_API_KEY.strip() == "":
        BPS_API_KEY = None
    print("BPS API key entered manually." if BPS_API_KEY else
          "No BPS API key provided; will use fallback CSV.")

# ------------------------------------------------------------------------------
# Manual Parameters CSV (Fallback / User Overrides)
# ------------------------------------------------------------------------------
MANUAL_PARAMS_FILE = 'rres_manual_parameters.csv'

def load_manual_parameters():
    params = {}
    if os.path.exists(MANUAL_PARAMS_FILE):
        with open(MANUAL_PARAMS_FILE, 'r') as f:
            reader = csv.reader(f)
            for row in reader:
                if len(row) == 2:
                    try:
                        params[row[0]] = float(row[1])
                    except ValueError:
                        params[row[0]] = row[1]
    return params

def save_manual_parameters(params):
    with open(MANUAL_PARAMS_FILE, 'w', newline='') as f:
        writer = csv.writer(f)
        for key, value in params.items():
            writer.writerow([key, value])

# ------------------------------------------------------------------------------
# BPS API: Trade Data (HS Code Aggregation)
# ------------------------------------------------------------------------------
def fetch_total_trade_for_year(api_key, trade_type, year):
    sumber = "1" if trade_type == "export" else "2"
    total = 0.0
    hs_codes = [f"{i:02d}" for i in range(1, 100)]
    print(f"Aggregating {trade_type} data for {year} "
          f"over {len(hs_codes)} HS codes...")
    for hs in hs_codes:
        params = {
            "sumber": sumber,
            "periode": "1",
            "kodehs": hs,
            "jenishs": "1",
            "tahun": str(year),
            "key": api_key
        }
        try:
            resp = requests.get(f"{BPS_BASE_URL}/dataexim",
                                params=params, timeout=10)
            if resp.status_code != 200:
                continue
            data = resp.json()
            if data.get("status") != "OK":
                continue
            items = data.get("data", [])
            for item in items:
                total += float(item.get("value", 0))
            time.sleep(0.05)
        except Exception:
            continue
    if total == 0:
        print(f"WARNING: Aggregation for {trade_type} returned zero")
        return None
    print(f"{trade_type.capitalize()} total: ${total:,.2f}")
    return total

def fetch_bps_trade_monthly(api_key, year=None):
    if not api_key:
        return None, None
    if year is None:
        year = datetime.datetime.now().year
    exports_annual = fetch_total_trade_for_year(api_key, "export", year)
    imports_annual = fetch_total_trade_for_year(api_key, "import", year)
    if exports_annual is None or imports_annual is None:
        return None, None
    exports_monthly = exports_annual / 12.0
    imports_monthly = imports_annual / 12.0
    return exports_monthly, imports_monthly

# ------------------------------------------------------------------------------
# World Bank API
# ------------------------------------------------------------------------------
def fetch_wb_inflation():
    try:
        data = wb.data.DataFrame('FP.CPI.TOTL.ZG', 'IDN',
                                 time=range(2015, 2027))
        if not data.empty:
            latest = data.iloc[-1].dropna()
            if len(latest) > 0:
                return latest.iloc[-1]
    except Exception:
        pass
    return None

def fetch_wb_indicator(indicator_code, country='IDN'):
    try:
        data = wb.data.DataFrame(indicator_code, country,
                                 time=range(2015, 2027))
        if not data.empty:
            latest = data.iloc[-1].dropna()
            if len(latest) > 0:
                return latest.iloc[-1]
    except Exception:
        pass
    return None

# ==============================================================================
# 1. FETCH LIVE DATA
# ==============================================================================

print("\nFetching live data...")
print("-" * 67)

# BPS Trade Data: API if available, otherwise fallback to repository CSV
USE_BPS_API = BPS_API_KEY is not None

if USE_BPS_API:
    print("Using BPS API for live trade data.")
    bps_exports_monthly, bps_imports_monthly = fetch_bps_trade_monthly(
        BPS_API_KEY
    )
else:
    print("No BPS API key found. Loading pre-fetched trade data.")
    csv_url = (
        "https://raw.githubusercontent.com/siregarmr/"
        "rupiah_reserve_exhaustion/main/bps_trade_latest.csv"
    )
    try:
        df_trade = pd.read_csv(csv_url)
        latest = df_trade.iloc[-1]
        bps_exports_monthly = float(latest["monthly_exports_usd"])
        bps_imports_monthly = float(latest["monthly_imports_usd"])
        print(f"Loaded data for year: {int(latest['year'])}")
    except Exception as e:
        print(f"ERROR: Could not load fallback trade data: {e}")
        bps_exports_monthly = None
        bps_imports_monthly = None

bps_inflation = fetch_wb_inflation()

if bps_exports_monthly and bps_imports_monthly:
    print("BPS Exports (monthly avg): "
          f"${bps_exports_monthly/1e9:.2f} billion")
    print("BPS Imports (monthly avg): "
          f"${bps_imports_monthly/1e9:.2f} billion")
else:
    print("ERROR: BPS trade data unavailable.")

if bps_inflation:
    print(f"Inflation (World Bank): {bps_inflation:.2f}%")
else:
    print("WARNING: Inflation data unavailable; using manual fallback.")

WB_INDICATORS = {
    'external_debt_usd': 'DT.DOD.DECT.CD',
    'gdp_usd': 'NY.GDP.MKTP.CD'
}
wb_data = {}
for key, code in WB_INDICATORS.items():
    val = fetch_wb_indicator(code)
    if val is not None and not np.isnan(val):
        wb_data[key] = val
        print(f"World Bank {key}: {val:,.2f}")
    else:
        wb_data[key] = None
        print(f"World Bank {key}: not available")

manual_params = load_manual_parameters()
print("\nManual parameters loaded: "
      f"{len(manual_params)} entries from {MANUAL_PARAMS_FILE}")

print("-" * 67)

if not bps_exports_monthly or not bps_imports_monthly:
    raise RuntimeError(
        "BPS trade data unavailable. Update manual parameters or fix API."
    )
else:
    print("BPS trade data fetched successfully.")

In [ ]:
# ==============================================================================
# 2. SIMULATION MODEL, PARAMETERS, STOCHASTIC ENGINE
# ==============================================================================

# ------------------------------------------------------------------------------
# Tunable Default Parameters (override via manual CSV)
# ------------------------------------------------------------------------------
DEFAULT_MANUAL = {
    'fx_reserves_usd': 148.2e9,
    'govt_external_debt_usd': 216.3e9,
    'monthly_debt_interest_usd': 300e6,
    'avg_debt_maturity_years': 8.0,
    'baseline_capital_outflow_usd': 500e6,
    'soft_burnout_months': 6.0,
    'hard_burnout_months': 3.0,
    'shock_prob_monthly': 0.05,
    'trade_shock_multiplier': 0.7,
    'outflow_shock_multiplier': 3.0,
    'inflation_shock_add': 2.0
}

for key, default_val in DEFAULT_MANUAL.items():
    if key not in manual_params:
        manual_params[key] = default_val

if bps_inflation is None:
    bps_inflation = manual_params.get('inflation_yoy_pct', 3.48)

# ------------------------------------------------------------------------------
# Final Parameter Dictionary
# ------------------------------------------------------------------------------
PARAMETERS = {
    'monthly_exports_usd': bps_exports_monthly,
    'monthly_imports_usd': bps_imports_monthly,
    'annual_exports_usd': bps_exports_monthly * 12,
    'annual_imports_usd': bps_imports_monthly * 12,
    'inflation_yoy_pct': bps_inflation,
    'fx_reserves_usd': manual_params['fx_reserves_usd'],
    'external_debt_total_usd': wb_data.get('external_debt_usd') or 434.7e9,
    'gdp_usd': wb_data.get('gdp_usd') or 1.47e12,
    'govt_external_debt_usd': manual_params['govt_external_debt_usd'],
    'monthly_debt_interest_usd': manual_params['monthly_debt_interest_usd'],
    'avg_debt_maturity_years': manual_params['avg_debt_maturity_years'],
    'baseline_capital_outflow_usd': manual_params[
        'baseline_capital_outflow_usd'
    ],
    'soft_burnout_months': manual_params['soft_burnout_months'],
    'hard_burnout_months': manual_params['hard_burnout_months'],
    'shock_prob_monthly': manual_params['shock_prob_monthly'],
    'trade_shock_multiplier': manual_params['trade_shock_multiplier'],
    'outflow_shock_multiplier': manual_params['outflow_shock_multiplier'],
    'inflation_shock_add': manual_params['inflation_shock_add'],
}

print("\n========= CURRENT SIMULATION PARAMETERS =========")
print("FX Reserves:              "
      f"${PARAMETERS['fx_reserves_usd']/1e9:.2f} billion")
print("Monthly Exports:          "
      f"${PARAMETERS['monthly_exports_usd']/1e9:.2f} billion")
print("Monthly Imports:          "
      f"${PARAMETERS['monthly_imports_usd']/1e9:.2f} billion")
print(f"Inflation (YoY):          {PARAMETERS['inflation_yoy_pct']:.2f}%")
print("Govt External Debt:       "
      f"${PARAMETERS['govt_external_debt_usd']/1e9:.2f} billion")
print("Monthly Debt Interest:    "
      f"${PARAMETERS['monthly_debt_interest_usd']/1e6:.2f} million")
print("Baseline Capital Outflow: "
      f"${PARAMETERS['baseline_capital_outflow_usd']/1e6:.2f} million/month")
print("Soft Burnout Threshold:   "
      f"{PARAMETERS['soft_burnout_months']} months import cover")
print("Hard Burnout Threshold:   "
      f"{PARAMETERS['hard_burnout_months']} months import cover")
print("=================================================\n")

# ------------------------------------------------------------------------------
# Simulation Control
# ------------------------------------------------------------------------------
SIM_YEARS = 15
N_SIMULATIONS = 10000
MONTHS = SIM_YEARS * 12

# Extract variables for faster access
FX_RESERVES_USD = PARAMETERS['fx_reserves_usd']
MONTHLY_IMPORTS = PARAMETERS['monthly_imports_usd']
MONTHLY_EXPORTS_BASE = PARAMETERS['monthly_exports_usd']
MONTHLY_DEBT_PRINCIPAL = (
    PARAMETERS['govt_external_debt_usd'] /
    (PARAMETERS['avg_debt_maturity_years'] * 12)
)
MONTHLY_DEBT_INTEREST_USD = PARAMETERS['monthly_debt_interest_usd']
BASELINE_CAPITAL_OUTFLOW_USD = PARAMETERS['baseline_capital_outflow_usd']
SOFT_BURNOUT_MONTHS = PARAMETERS['soft_burnout_months']
HARD_BURNOUT_MONTHS = PARAMETERS['hard_burnout_months']
SHOCK_PROB_MONTHLY = PARAMETERS['shock_prob_monthly']
TRADE_SHOCK_MULTIPLIER = PARAMETERS['trade_shock_multiplier']
OUTFLOW_SHOCK_MULTIPLIER = PARAMETERS['outflow_shock_multiplier']
INFLATION_SHOCK_ADD = PARAMETERS['inflation_shock_add']
INFLATION_YOY_PCT = PARAMETERS['inflation_yoy_pct']

# ------------------------------------------------------------------------------
# Stochastic Simulation Function
# ------------------------------------------------------------------------------
def simulate_monthly_reserves(
    months, init_reserves, monthly_imports, monthly_exports_base,
    monthly_debt_principal, monthly_debt_interest, baseline_outflow
):
    reserves = np.zeros(months + 1)
    reserves[0] = init_reserves
    exports = monthly_exports_base
    capital_outflow = baseline_outflow
    sigma_trade = 0.03 * monthly_exports_base
    sigma_outflow = 0.05 * baseline_outflow
    burnout_time_soft = np.inf
    burnout_time_hard = np.inf

    for m in range(1, months + 1):
        if np.random.random() < SHOCK_PROB_MONTHLY:
            exports = monthly_exports_base * TRADE_SHOCK_MULTIPLIER
            capital_outflow = baseline_outflow * OUTFLOW_SHOCK_MULTIPLIER
        else:
            exports = exports * 0.8 + monthly_exports_base * 0.2
            capital_outflow = (
                capital_outflow * 0.8 + baseline_outflow * 0.2
            )
        exports += np.random.normal(0, sigma_trade)
        capital_outflow += np.random.normal(0, sigma_outflow)
        exports = max(exports, monthly_exports_base * 0.5)
        capital_outflow = max(capital_outflow, 0)

        trade_balance = exports - monthly_imports
        debt_service = monthly_debt_principal + monthly_debt_interest
        net_change = trade_balance - debt_service - capital_outflow

        reserves[m] = max(reserves[m-1] + net_change, 0)

        months_cover = (
            reserves[m] / monthly_imports if monthly_imports > 0 else np.inf
        )

        if months_cover < SOFT_BURNOUT_MONTHS and np.isinf(burnout_time_soft):
            burnout_time_soft = m
        if months_cover < HARD_BURNOUT_MONTHS and np.isinf(burnout_time_hard):
            burnout_time_hard = m

    import_cover = reserves / monthly_imports
    return reserves, import_cover, burnout_time_soft, burnout_time_hard

print("Simulation engine defined.\n")

In [ ]:
# ==============================================================================
# 3. RUN MONTE CARLO SIMULATION
# ==============================================================================

print(f"Running {N_SIMULATIONS} scenarios over {SIM_YEARS} years...")
print("This may take 30-60 seconds...\n")

burnout_times_soft = np.full(N_SIMULATIONS, np.inf)
burnout_times_hard = np.full(N_SIMULATIONS, np.inf)
reserve_paths = np.zeros((N_SIMULATIONS, MONTHS + 1))
cover_paths = np.zeros((N_SIMULATIONS, MONTHS + 1))

for sim in range(N_SIMULATIONS):
    if sim % 1000 == 0:
        print(f"Progress: {sim}/{N_SIMULATIONS}")
    res, cover, soft, hard = simulate_monthly_reserves(
        MONTHS, FX_RESERVES_USD, MONTHLY_IMPORTS,
        MONTHLY_EXPORTS_BASE, MONTHLY_DEBT_PRINCIPAL,
        MONTHLY_DEBT_INTEREST_USD, BASELINE_CAPITAL_OUTFLOW_USD
    )
    reserve_paths[sim, :] = res
    cover_paths[sim, :] = cover
    burnout_times_soft[sim] = soft
    burnout_times_hard[sim] = hard

burnout_years_soft = burnout_times_soft / 12
burnout_years_hard = burnout_times_hard / 12

# ------------------------------------------------------------------------------
# Results Summary
# ------------------------------------------------------------------------------
horizons = [5, 10, 15]
print("\n==== SIMULATION RESULTS ====")
for h in horizons:
    months_h = h * 12
    prob_soft = np.mean(burnout_times_soft <= months_h)
    prob_hard = np.mean(burnout_times_hard <= months_h)
    print(f"{h} years:")
    print(f"  Soft Burnout (6mo): {prob_soft:.1%}")
    print(f"  Hard Burnout (3mo): {prob_hard:.1%}")

if np.any(burnout_times_soft < np.inf):
    med_soft = np.median(burnout_years_soft[burnout_years_soft < np.inf])
    print(f"\nMedian time to Soft Burnout: {med_soft:.2f} years")
if np.any(burnout_times_hard < np.inf):
    med_hard = np.median(burnout_years_hard[burnout_years_hard < np.inf])
    print(f"Median time to Hard Burnout: {med_hard:.2f} years")

In [ ]:
# ==============================================================================
# 4. VISUALIZATION
# ==============================================================================

import matplotlib.pyplot as plt
%matplotlib inline

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
time_years = np.arange(MONTHS + 1) / 12

# ------------------------------------------------------------------------------
# Subplot 1: FX Reserve Levels
# ------------------------------------------------------------------------------
ax1 = axes[0, 0]
p10_res = np.percentile(reserve_paths, 10, axis=0) / 1e9
p50_res = np.percentile(reserve_paths, 50, axis=0) / 1e9
p90_res = np.percentile(reserve_paths, 90, axis=0) / 1e9
ax1.fill_between(time_years, p10_res, p90_res, alpha=0.3,
                 color='blue', label='10th-90th Percentile')
ax1.plot(time_years, p50_res, 'b-', linewidth=2, label='Median Reserves')
ax1.axhline(FX_RESERVES_USD/1e9, color='gray', linestyle=':',
            label=f'Initial (${FX_RESERVES_USD/1e9:.1f}B)')
ax1.set_xlabel('Years from Now')
ax1.set_ylabel('FX Reserves (USD Billion)')
ax1.set_title('Projected FX Reserve Levels')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# ------------------------------------------------------------------------------
# Subplot 2: Import Cover
# ------------------------------------------------------------------------------
ax2 = axes[0, 1]
p10_cov = np.percentile(cover_paths, 10, axis=0)
p50_cov = np.percentile(cover_paths, 50, axis=0)
p90_cov = np.percentile(cover_paths, 90, axis=0)
ax2.fill_between(time_years, p10_cov, p90_cov, alpha=0.3,
                 color='orange', label='10th-90th Percentile')
ax2.plot(time_years, p50_cov, 'orange', linewidth=2,
         label='Median Import Cover')
ax2.axhline(SOFT_BURNOUT_MONTHS, color='red', linestyle='--',
            label=f'Soft Burnout ({SOFT_BURNOUT_MONTHS} mo)')
ax2.axhline(HARD_BURNOUT_MONTHS, color='darkred', linestyle='-',
            label=f'Hard Burnout ({HARD_BURNOUT_MONTHS} mo)')
ax2.set_xlabel('Years from Now')
ax2.set_ylabel('Import Cover (Months)')
ax2.set_title('Projected Reserve Adequacy')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

# ------------------------------------------------------------------------------
# Subplot 3: Soft Burnout Time Distribution
# ------------------------------------------------------------------------------
ax3 = axes[1, 0]
soft_years_occ = burnout_years_soft[burnout_years_soft < np.inf]
if len(soft_years_occ) > 0:
    ax3.hist(soft_years_occ, bins=30, color='gold', alpha=0.7,
             edgecolor='black')
    ax3.axvline(np.median(soft_years_occ), color='red', linestyle='--',
                label=f'Median: {np.median(soft_years_occ):.2f} yrs')
    ax3.set_xlabel('Years until Soft Burnout')
    ax3.set_ylabel('Frequency')
    ax3.set_title(f'Soft Burnout Timing (N={len(soft_years_occ):,})')
    ax3.legend()
else:
    ax3.text(0.5, 0.5, 'No Soft Burnouts Simulated',
             ha='center', va='center', transform=ax3.transAxes, fontsize=12)
ax3.grid(True, alpha=0.3)

# ------------------------------------------------------------------------------
# Subplot 4: Cumulative Burnout Probability
# ------------------------------------------------------------------------------
ax4 = axes[1, 1]
cum_soft = np.array([np.mean(burnout_times_soft <= m) for m in range(MONTHS+1)])
cum_hard = np.array([np.mean(burnout_times_hard <= m) for m in range(MONTHS+1)])
ax4.plot(time_years, cum_soft, 'orange', linewidth=2,
         label='Soft Burnout (6mo)')
ax4.plot(time_years, cum_hard, 'red', linewidth=2,
         label='Hard Burnout (3mo)')
ax4.set_xlabel('Years from Now')
ax4.set_ylabel('Cumulative Probability')
ax4.set_title('Cumulative Burnout Probability Over Time')
ax4.legend(loc='lower right')
ax4.grid(True, alpha=0.3)
ax4.set_ylim(0, 1)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------------------------
# Annual hazard rates
# ------------------------------------------------------------------------------
print("\n=== ANNUAL BURNOUT PROBABILITY ===")
for year in range(1, min(SIM_YEARS, 15) + 1):
    months_start = (year - 1) * 12
    months_end = year * 12
    prob_soft_year = np.mean(
        (burnout_times_soft > months_start) &
        (burnout_times_soft <= months_end)
    )
    prob_hard_year = np.mean(
        (burnout_times_hard > months_start) &
        (burnout_times_hard <= months_end)
    )
    print(f"Year {year:2d}:  Soft Burnout {prob_soft_year:.2%}, "
          f"Hard Burnout {prob_hard_year:.2%}")